# DeepfakeBench — EfficientNet-B4 baseline on Kaggle (FF++ train, Celeb-DF-v2 cross-test)

**Setup before running:** GPU T4 x1 + Internet ON.

**Data attachment options** (Add data → Search dataset → Add):

| Need | Recommended Kaggle Dataset | Size | Slug |
|---|---|---|---|
| FF++ c23 videos | `xdxd003/ff-c23` | ~38 GB | `/kaggle/input/ff-c23` |
| FF++ c23 frames (alt, already extracted) | `fatimahirshad/faceforensics-extracted-dataset-c23` | smaller | `/kaggle/input/faceforensics-extracted-dataset-c23` |
| Celeb-DF-v2 | `reubensuju/celeb-df-v2` or `shreyanshmanavshukla/celebsv2-faces-224` | ~30 GB | `/kaggle/input/celeb-df-v2` |

If a dataset already has extracted face frames (224 or 256), you can SKIP the face-extract step and jump to **Step 4** with adapted paths.

**Phases**
1. Clone & install
2. (Optional) Extract face crops from raw videos → saves to `/kaggle/working/processed`
3. Build DeepfakeBench JSON manifests
4. Apply config + smoke test (1 epoch, 8 frames)
5. Full train + resume strategy
6. Standalone cross-test on Celeb-DF-v2

⚠️ Realistic timeline: face extraction alone is ~3-6h for FF++ c23. Use Kaggle persistent output to save processed frames, then re-attach in next session for training.

## 1. Environment + clone DeepfakeBench

In [ ]:
!nvidia-smi | head -15
!free -h
!df -h /kaggle/working

In [ ]:
import os
os.chdir('/kaggle/working')
if not os.path.exists('DeepfakeBench'):
    !git clone https://github.com/huanthuytnhh/DeepfakeBench.git
os.chdir('/kaggle/working/DeepfakeBench')
!git log --oneline | head -5

In [ ]:
!pip install -q efficientnet_pytorch==0.7.1 albumentations==1.1.0 imutils==0.5.4 \
    timm==0.6.12 einops==0.4.1 transformers==4.27.4 \
    filterpy lmdb opencv-python kornia==0.6.5 imgaug==0.4.0 \
    fvcore simplejson scikit-image scikit-learn \
    facenet-pytorch==2.5.3
import torch; print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available(), 'device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

In [ ]:
# Download EfficientNet-B4 ImageNet pretrained weights
import urllib.request
os.makedirs('training/pretrained', exist_ok=True)
url = 'https://github.com/lukemelas/EfficientNet-PyTorch/releases/download/1.0/efficientnet-b4-6ed6700e.pth'
out = 'training/pretrained/efficientnet-b4-6ed6700e.pth'
if not os.path.exists(out):
    urllib.request.urlretrieve(url, out)
print(os.path.getsize(out)/1e6, 'MB downloaded')

## 2. Extract face crops from raw videos (skip if data already has face crops)

**Adjust `FFPP_VIDEO_ROOT` and `CDF_VIDEO_ROOT` to match your attached datasets.**

The script writes face crops to `/kaggle/working/processed/<DatasetName>/...` in the exact folder layout DeepfakeBench expects.

If your Kaggle Dataset has a different layout, do a `!find /kaggle/input -maxdepth 4 -type d | head -50` first to inspect.

In [ ]:
# Inspect attached input datasets
!ls /kaggle/input/ 2>/dev/null
print('---')
!find /kaggle/input -maxdepth 3 -type d 2>/dev/null | head -40

In [ ]:
# === EDIT THESE PATHS ===
FFPP_VIDEO_ROOT = '/kaggle/input/ff-c23'         # must contain subfolders: youtube/, Deepfakes/, Face2Face/, FaceSwap/, NeuralTextures/
CDF_VIDEO_ROOT  = '/kaggle/input/celeb-df-v2'    # must contain: Celeb-real/, Celeb-synthesis/, YouTube-real/, List_of_testing_videos.txt
PROCESSED_ROOT  = '/kaggle/working/processed'
NUM_FRAMES      = 32

import os
os.makedirs(PROCESSED_ROOT, exist_ok=True)

In [ ]:
# Extract FF++ face crops (slow: ~3-6h for full 5000 videos x 32 frames)
# For smoke test, set DRY_RUN_LIMIT to extract only first N videos per category
DRY_RUN_LIMIT = 50  # set to None for full extraction

if DRY_RUN_LIMIT is None:
    !python tools/extract_faces_from_videos.py \
        --dataset FaceForensics++ \
        --video_root {FFPP_VIDEO_ROOT} \
        --out_root {PROCESSED_ROOT} \
        --num_frames {NUM_FRAMES}
else:
    print(f'DRY_RUN_LIMIT={DRY_RUN_LIMIT} — skipping full extract. Implement subset extraction by limiting walk_videos() if needed.')
    # For real smoke test, just run on a small subset of videos manually

In [ ]:
# Extract Celeb-DF-v2 face crops (~1-2h for full)
RUN_CDF = True
if RUN_CDF:
    !python tools/extract_faces_from_videos.py \
        --dataset Celeb-DF-v2 \
        --video_root {CDF_VIDEO_ROOT} \
        --out_root {PROCESSED_ROOT} \
        --num_frames {NUM_FRAMES}

## 3. Build DeepfakeBench JSON manifests

Reads `/kaggle/working/processed/*` and writes `preprocessing/dataset_json_v3/*.json` (the folder name `dataset_json_v3` is required by `train.py:245`).

In [ ]:
!python tools/build_deepfakebench_json.py \
    --rgb_root /kaggle/working/processed \
    --output_dir preprocessing/dataset_json_v3 \
    --datasets FaceForensics++ Celeb-DF-v2

!ls -la preprocessing/dataset_json_v3/

## 4. Apply detector config + train_config (paths) + smoke test

In [ ]:
# Update train_config.yaml to point at /kaggle/working/processed (RGB) — no LMDB
train_cfg = f'''mode: train
lmdb: False
dry_run: false
rgb_dir: '/kaggle/working/processed'
lmdb_dir: '/kaggle/working/lmdb'
dataset_json_folder: 'preprocessing/dataset_json_v3'
SWA: False
save_avg: True
log_dir: /kaggle/working/logs/training/
label_dict:
  DFD_fake: 1
  DFD_real: 0
  FF-SH: 1
  FF-F2F: 1
  FF-DF: 1
  FF-FS: 1
  FF-NT: 1
  FF-FH: 1
  FF-real: 0
  CelebDFv1_real: 0
  CelebDFv1_fake: 1
  CelebDFv2_real: 0
  CelebDFv2_fake: 1
  DFDCP_Real: 0
  DFDCP_FakeA: 1
  DFDCP_FakeB: 1
  DFDC_Fake: 1
  DFDC_Real: 0
  DF_fake: 1
  DF_real: 0
  UADFV_Fake: 1
  UADFV_Real: 0
  roop_Real: 0
  roop_Fake: 1
'''
with open('training/config/train_config.yaml', 'w') as f:
    f.write(train_cfg)
with open('training/config/test_config.yaml', 'w') as f:
    f.write(train_cfg.replace('mode: train', 'mode: test'))
!grep -E 'lmdb|rgb_dir|dataset_json' training/config/train_config.yaml

In [ ]:
# IMPORTANT: When lmdb=False, train.py:245 still rewrites dataset_json_folder if 'lmdb' truthy. Verify:
# Actually train.py:244 checks `if config['lmdb']:` so when lmdb=False the rewrite is skipped — good.
# But abstract_dataset.py:298-299 prepends './<rgb_dir>\\' to file_path. Linux uses /, so we patch:
import re
p = 'training/dataset/abstract_dataset.py'
s = open(p).read()
# Replace Windows backslash join in 3 places with forward slash, and skip prepend if already absolute
s2 = s.replace("file_path =  f'./{self.config[\"rgb_dir\"]}\\\\'+file_path",
               "file_path = file_path if file_path.startswith('/') else os.path.join(self.config['rgb_dir'], file_path)")
if s != s2:
    open(p, 'w').write(s2)
    print('Patched abstract_dataset.py for Linux paths')
else:
    print('No change needed (already patched or pattern not found)')

In [ ]:
# Smoke test: 1 epoch, 8 frames, batch 4 (~10 min if data is small)
import yaml
with open('training/config/detector/efficientnetb4.yaml') as f:
    cfg = yaml.safe_load(f)
cfg['nEpochs'] = 1
cfg['frame_num'] = {'train': 8, 'test': 8}
cfg['train_batchSize'] = 4
cfg['test_batchSize'] = 4
cfg['log_dir'] = '/kaggle/working/logs/evaluations/effnb4_smoke'
with open('training/config/detector/efficientnetb4.yaml', 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

!cd /kaggle/working/DeepfakeBench && \
  PYTHONPATH=./training python training/train.py \
    --detector_path ./training/config/detector/efficientnetb4.yaml \
    --train_dataset FaceForensics++ \
    --test_dataset FaceForensics++ Celeb-DF-v2 \
    --task_target smoke 2>&1 | tee /kaggle/working/smoke.log

**Smoke pass criteria:**
- ✅ `train: N images, M videos` (N > 0)
- ✅ Loss giảm qua iter
- ✅ Eval AUC ra số
- ✅ Checkpoint xuất hiện trong `/kaggle/working/logs/evaluations/effnb4_smoke/...`

Nếu fail → đọc `/kaggle/working/smoke.log`, KHÔNG chạy full train.

## 5. Full train (only after smoke passes)

In [ ]:
import yaml
with open('training/config/detector/efficientnetb4.yaml') as f:
    cfg = yaml.safe_load(f)
cfg['nEpochs'] = 10
cfg['frame_num'] = {'train': 32, 'test': 32}
cfg['train_batchSize'] = 16
cfg['test_batchSize'] = 16
cfg['log_dir'] = '/kaggle/working/logs/evaluations/effnb4'
with open('training/config/detector/efficientnetb4.yaml', 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

!cd /kaggle/working/DeepfakeBench && \
  PYTHONPATH=./training python training/train.py \
    --detector_path ./training/config/detector/efficientnetb4.yaml \
    --train_dataset FaceForensics++ \
    --test_dataset FaceForensics++ Celeb-DF-v2 \
    --task_target ffpp_baseline 2>&1 | tee /kaggle/working/train.log

## 6. Resume after Kaggle session timeout (12h)

Run cells 1–4 again in the new session, then this cell:

In [ ]:
import glob, yaml
ckpts = sorted(glob.glob('/kaggle/working/logs/evaluations/effnb4/*/ckpt_best.pth'))
if not ckpts:
    print('No checkpoint found — start fresh.')
else:
    last_ckpt = ckpts[-1]
    print('Resume from:', last_ckpt)
    with open('training/config/detector/efficientnetb4.yaml') as f:
        cfg = yaml.safe_load(f)
    cfg['pretrained'] = last_ckpt
    cfg['start_epoch'] = 5   # adjust: number of epochs already completed (check train.log of prev session)
    cfg['nEpochs'] = 10
    with open('training/config/detector/efficientnetb4.yaml', 'w') as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    !cd /kaggle/working/DeepfakeBench && \
      PYTHONPATH=./training python training/train.py \
        --detector_path ./training/config/detector/efficientnetb4.yaml \
        --train_dataset FaceForensics++ \
        --test_dataset FaceForensics++ Celeb-DF-v2 \
        --task_target ffpp_baseline_resume

## 7. Cross-test on Celeb-DF-v2 with best checkpoint

In [ ]:
import glob
ckpts = sorted(glob.glob('/kaggle/working/logs/evaluations/effnb4/*/ckpt_best.pth'))
BEST_CKPT = ckpts[-1] if ckpts else None
print('Best ckpt:', BEST_CKPT)
if BEST_CKPT:
    !cd /kaggle/working/DeepfakeBench && \
      PYTHONPATH=./training python training/test.py \
        --detector_path ./training/config/detector/efficientnetb4.yaml \
        --test_dataset Celeb-DF-v2 \
        --weights_path {BEST_CKPT}

## 8. Save artifacts (download from Kaggle output)

In [ ]:
import shutil, glob, os
src = '/kaggle/working/logs/evaluations/effnb4'
runs = sorted(glob.glob(f'{src}/*'))
print('Runs:', runs)
if runs:
    shutil.make_archive('/kaggle/working/effnb4_result', 'zip', runs[-1])
    print('Zip size:', os.path.getsize('/kaggle/working/effnb4_result.zip')/1e6, 'MB')

---

## Common pitfalls and fixes

| Symptom | Fix |
|---|---|
| `dataset FaceForensics++ not exist!` | Missing `preprocessing/dataset_json_v3/FaceForensics++.json`. Re-run Step 3. |
| `KeyError: 'CelebDFv2_fake'` in label_dict | Verify label keys in `Celeb-DF-v2.json` match `train_config.yaml` exactly. |
| `FileNotFoundError: train.json` (FF++ split) | Run `tools/build_deepfakebench_json.py` which auto-downloads from FaceForensics GitHub. |
| OOM at batch 16 | Lower `train_batchSize` to 8 (record this for reproducibility). |
| Loss NaN early | Lower lr to 0.0001 in detector YAML. |
| Mid-training Kaggle timeout | Use Step 6 resume. Always `save_epoch: 1` so a checkpoint per epoch survives. |
| `FF++ frames dir empty` | Your video root subfolder names differ (e.g. `original_sequences/youtube/c23/videos` vs `youtube/`). Inspect `find /kaggle/input -maxdepth 4 -type d` and adjust extraction script. |

## Expected results (paper benchmark)

| Train | Test (within FF++) | Test (cross Celeb-DF-v2) |
|---|---|---|
| FF++ c23 (10 epoch, EfficientNet-B4) | AUC ~0.95 | AUC ~0.74-0.80 |

If your Celeb-DF cross-test AUC < 0.65 → data manifest or label mapping is wrong; verify a handful of frames + labels manually before assuming model failure.